# Basic Quantum Error Correction

This notebook is learning material for **basic quantum error correction**.

It explains:

- **why** quantum error correction is needed,
- **what kinds of errors happen**,
- **how simple correction works**,
- **what a syndrome is**,
- **where quantum error correction is used**.

The examples are intentionally simple and intuitive.

## 1. Why do we need quantum error correction?

A qubit is not just a symbol on paper.  
It is stored in a real physical system, such as:

- an ion,
- a photon,
- an atom,
- an electron spin,
- a superconducting circuit.

Because the qubit is physical, it can interact with its environment.

That causes **noise**.

Noise can accidentally change the quantum state.

Examples:

$
|0\rangle \leftrightarrow |1\rangle
$

This is called a **bit-flip error**.

Another type of error changes the **phase**:

$
|+\rangle \leftrightarrow |-\rangle
$

This is called a **phase-flip error**.

A quantum computer may need millions or billions of operations.  
Even tiny error rates become a problem if we do many operations.

So quantum error correction is needed because:

$
\text{real qubits are noisy}
$

$
\text{long computations need many gates}
$

$
\text{small errors accumulate}
$

## 2. Why quantum error correction is harder than classical error correction

Classical error correction can copy information.

For example, instead of storing one bit:

$
0
$

we store:

$
000
$

If one bit flips:

$
000 \rightarrow 010
$

we use majority vote and recover \(0\).

But quantum information is different.

### Problem 1: We cannot copy an unknown qubit

An unknown qubit can be

$
|\psi\rangle = a|0\rangle + b|1\rangle.
$

There is no allowed quantum operation that simply copies it into

$
|\psi\rangle|\psi\rangle.
$

This is called the **no-cloning principle**.

### Problem 2: Measuring directly destroys the state

If we directly measure

$
|\psi\rangle = a|0\rangle + b|1\rangle,
$

we get only \(0\) or \(1\).  
The original superposition is destroyed.

So quantum error correction must do something clever:

$
\text{detect the error without measuring the protected quantum state.}
$

## 3. Setup

Run this cell first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import comb

try:
    import ipywidgets as widgets
    from ipywidgets import interact
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False
    print("ipywidgets is not available. The notebook still works without sliders.")

np.set_printoptions(precision=3, suppress=True)

# Basic one-qubit states
ket0 = np.array([1, 0], dtype=complex)
ket1 = np.array([0, 1], dtype=complex)
ket_plus = (ket0 + ket1) / np.sqrt(2)
ket_minus = (ket0 - ket1) / np.sqrt(2)

# Basic one-qubit gates
I = np.eye(2, dtype=complex)
X = np.array([[0, 1],
              [1, 0]], dtype=complex)
Z = np.array([[1, 0],
              [0, -1]], dtype=complex)
H = (1 / np.sqrt(2)) * np.array([[1, 1],
                                  [1, -1]], dtype=complex)

def kron(*items):
    result = np.asarray(items[0], dtype=complex)
    for item in items[1:]:
        result = np.kron(result, item)
    return result

def basis_labels(n):
    return [f"|{i:0{n}b}⟩" for i in range(2**n)]

def basis_state(bitstring):
    state = np.zeros(2**len(bitstring), dtype=complex)
    state[int(bitstring, 2)] = 1
    return state

def normalize(state):
    norm = np.linalg.norm(state)
    if norm == 0:
        raise ValueError("Cannot normalize the zero vector.")
    return state / norm

def print_state(state, labels=None, tol=1e-10):
    if labels is None:
        labels = [str(i) for i in range(len(state))]
    for amp, label in zip(state, labels):
        if abs(amp) > tol:
            print(f"{label}: {amp:.3g}")
    print("total probability:", round(float(np.sum(np.abs(state)**2)), 6))

def show_probabilities(state, labels=None, title="Probabilities"):
    probs = np.abs(state)**2
    if labels is None:
        labels = [str(i) for i in range(len(probs))]
    plt.figure(figsize=(max(5, len(labels)*0.65), 3))
    plt.bar(labels, probs)
    plt.ylim(0, 1)
    plt.ylabel("Probability")
    plt.title(title)
    plt.show()

def gate_on_n_qubits(U, n, target):
    # target 0 means the leftmost qubit
    ops = [U if i == target else I for i in range(n)]
    return kron(*ops)

def apply_X_error(state, n, target):
    return gate_on_n_qubits(X, n, target) @ state

def apply_Z_error(state, n, target):
    return gate_on_n_qubits(Z, n, target) @ state

## 4. Classical warm-up: repetition code

Before quantum error correction, look at classical repetition.

We encode:

$
0 \rightarrow 000
$

$
1 \rightarrow 111
$

If one bit flips, majority vote can fix it.

In [ ]:
def majority_vote(bits):
    if bits.count("1") > bits.count("0"):
        return "1"
    return "0"

def classical_repetition_demo(original_bit="0", error_position=1):
    encoded = original_bit * 3

    corrupted = list(encoded)
    corrupted[error_position] = "1" if corrupted[error_position] == "0" else "0"
    corrupted = "".join(corrupted)

    decoded = majority_vote(corrupted)

    print("original bit:", original_bit)
    print("encoded:     ", encoded)
    print("corrupted:   ", corrupted)
    print("decoded:     ", decoded)
    print("recovered correctly:", decoded == original_bit)

if WIDGETS_AVAILABLE:
    interact(
        classical_repetition_demo,
        original_bit=widgets.Dropdown(options=["0", "1"], value="0", description="bit"),
        error_position=widgets.IntSlider(value=1, min=0, max=2, description="flip pos"),
    )
else:
    classical_repetition_demo()

## 5. Why redundancy helps

If one physical bit fails with probability \(p\), an unprotected bit fails with probability

$
p.
$

A 3-bit repetition code fails only if two or three bits flip:

$
P_{\text{fail}} = 3p^2(1-p)+p^3.
$

For small \(p\), this is much smaller than \(p\).

This is the basic protection idea:

$
\text{use many physical systems to protect one logical system.}
$

In [ ]:
def repetition_failure_probability(n, p):
    # majority vote fails if more than half the bits flip
    fail = 0
    for k in range(n//2 + 1, n + 1):
        fail += comb(n, k) * (p**k) * ((1-p)**(n-k))
    return fail

def plot_repetition_failure(p=0.05):
    ns = [1, 3, 5, 7, 9]
    failures = [repetition_failure_probability(n, p) for n in ns]

    plt.figure(figsize=(6, 3))
    plt.bar([str(n) for n in ns], failures)
    plt.xlabel("number of physical bits")
    plt.ylabel("failure probability")
    plt.title(f"Majority-vote failure probability when p = {p:.2f}")
    plt.show()

    for n, failure in zip(ns, failures):
        print(f"{n} physical bits -> failure probability ≈ {failure:.6f}")

if WIDGETS_AVAILABLE:
    interact(
        plot_repetition_failure,
        p=widgets.FloatSlider(value=0.05, min=0, max=0.5, step=0.01, description="error p"),
    )
else:
    plot_repetition_failure()

## 6. The 3-qubit quantum bit-flip code

Now we protect a qubit against one bit-flip error.

Logical states:

$
|0_L\rangle = |000\rangle
$

$
|1_L\rangle = |111\rangle
$

A general encoded logical qubit is

$
|\psi_L\rangle = a|000\rangle + b|111\rangle.
$

This is not copying the qubit.  
It is encoding the information into a bigger quantum state.

In [ ]:
def encode_bit_flip_code(a, b):
    state = a * basis_state("000") + b * basis_state("111")
    return normalize(state)

def show_encoded_state(a=1.0, b=1.0):
    encoded = encode_bit_flip_code(a, b)
    print("encoded logical qubit:")
    print_state(encoded, basis_labels(3))
    show_probabilities(encoded, basis_labels(3), "Encoded logical qubit")

if WIDGETS_AVAILABLE:
    interact(
        show_encoded_state,
        a=widgets.FloatSlider(value=1.0, min=-1.0, max=1.0, step=0.05, description="a"),
        b=widgets.FloatSlider(value=1.0, min=-1.0, max=1.0, step=0.05, description="b"),
    )
else:
    show_encoded_state()

## 7. What is a syndrome?

A **syndrome** is information about the error.

It tells us which physical qubit looks wrong.

It does **not** tell us the protected state \(a|000\rangle+b|111\rangle\).

For a 3-qubit bit-flip code, we compare neighboring qubits:

$
s_1 = q_0 \oplus q_1
$

$
s_2 = q_1 \oplus q_2
$

The syndrome tells us the correction:

| Syndrome | Meaning |
|---|---|
| `00` | no bit-flip detected |
| `11` | qubit 0 flipped |
| `10` | qubit 1 flipped |
| `01` | qubit 2 flipped |

This is the central trick of quantum error correction:

$
\text{measure the error, not the logical quantum information.}
$

In [ ]:
def bit_flip_syndrome(bitstring):
    q0, q1, q2 = [int(x) for x in bitstring]
    s1 = q0 ^ q1
    s2 = q1 ^ q2
    return f"{s1}{s2}"

def correction_from_syndrome(syndrome):
    table = {
        "00": None,
        "11": 0,
        "10": 1,
        "01": 2,
    }
    return table[syndrome]

def syndrome_demo(bitstring="010"):
    syndrome = bit_flip_syndrome(bitstring)
    correction = correction_from_syndrome(syndrome)

    print("observed bitstring:", bitstring)
    print("syndrome:", syndrome)
    print("correction qubit:", correction)

if WIDGETS_AVAILABLE:
    interact(
        syndrome_demo,
        bitstring=widgets.Dropdown(
            options=["000", "100", "010", "001", "111", "011", "101", "110"],
            value="010",
            description="bits",
        ),
    )
else:
    syndrome_demo()

## 8. Correcting one bit-flip error

The bit-flip code can correct **one** \(X\) error.

Process:

1. encode the logical qubit,
2. one physical qubit flips,
3. read the syndrome,
4. apply \(X\) to the corrupted qubit,
5. recover the encoded logical state.

In [ ]:
def bit_flip_correction_demo(error_qubit=1):
    encoded = encode_bit_flip_code(1, 1)
    corrupted = apply_X_error(encoded, 3, error_qubit)

    labels = basis_labels(3)

    # Pick one nonzero basis string to infer the syndrome.
    nonzero_bitstrings = [
        label.strip("|⟩")
        for amp, label in zip(corrupted, labels)
        if abs(amp) > 1e-10
    ]
    sample_bitstring = nonzero_bitstrings[0]

    syndrome = bit_flip_syndrome(sample_bitstring)
    correction = correction_from_syndrome(syndrome)

    if correction is None:
        corrected = corrupted
    else:
        corrected = apply_X_error(corrupted, 3, correction)

    print("actual error qubit:", error_qubit)
    print("detected syndrome:", syndrome)
    print("chosen correction:", correction)

    print("\ncorrupted state:")
    print_state(corrupted, labels)

    print("\ncorrected state:")
    print_state(corrected, labels)

    print("\nrecovered original encoded state:", np.allclose(corrected, encoded))

if WIDGETS_AVAILABLE:
    interact(
        bit_flip_correction_demo,
        error_qubit=widgets.IntSlider(value=1, min=0, max=2, description="error qubit"),
    )
else:
    bit_flip_correction_demo()

## 9. Phase errors

Quantum states also have **phase**.

A phase-flip error is caused by the \(Z\) gate.

$
Z|+\rangle = |-\rangle
$

$
Z|-\rangle = |+\rangle
$

A phase error may not be visible immediately in the \(0/1\) measurement basis.  
But after a Hadamard gate, it becomes visible:

$
H|+\rangle = |0\rangle
$

$
H|-\rangle = |1\rangle
$

This is why quantum error correction must protect both:

- bit errors,
- phase errors.

In [ ]:
print("H Z H equals:")
print(np.round(H @ Z @ H, 3))

print("\nThis is the X gate:")
print(X)

def phase_error_demo(apply_error=True):
    state = ket_plus

    if apply_error:
        state = Z @ state

    final = H @ state

    print("applied Z phase error:", apply_error)
    print_state(final, ["|0⟩", "|1⟩"])
    show_probabilities(final, ["0", "1"], "After final H")

if WIDGETS_AVAILABLE:
    interact(
        phase_error_demo,
        apply_error=widgets.Checkbox(value=True, description="apply Z error"),
    )
else:
    phase_error_demo()

## 10. Simple noise simulation

The 3-qubit bit-flip code can correct one bit-flip error.

It fails when two or three physical qubits flip.

This simulation compares:

$
\text{one physical qubit error probability}
$

with

$
\text{protected logical failure probability}.
$

In [ ]:
rng = np.random.default_rng(7)

def simulate_three_qubit_code(p=0.05, trials=10000):
    failures = 0

    for _ in range(trials):
        flips = rng.random(3) < p
        number_of_flips = np.sum(flips)

        if number_of_flips >= 2:
            failures += 1

    return failures / trials

def compare_physical_and_logical(p=0.05):
    physical_error = p
    logical_formula = repetition_failure_probability(3, p)
    logical_simulation = simulate_three_qubit_code(p)

    print("physical error probability:", physical_error)
    print("logical error formula:", logical_formula)
    print("logical error simulation:", logical_simulation)

    plt.figure(figsize=(5, 3))
    plt.bar(["physical qubit", "protected logical qubit"], [physical_error, logical_formula])
    plt.ylabel("failure probability")
    plt.title("Error correction helps when physical errors are rare")
    plt.show()

if WIDGETS_AVAILABLE:
    interact(
        compare_physical_and_logical,
        p=widgets.FloatSlider(value=0.05, min=0, max=0.5, step=0.01, description="error p"),
    )
else:
    compare_physical_and_logical()

## 11. Where is quantum error correction used?

Quantum error correction is used in the design of **fault-tolerant quantum computers**.

It is important for:

### 1. Long quantum algorithms

Algorithms that require many gates need protection from accumulated errors.

### 2. Logical qubits

A useful quantum computer will not usually compute directly with raw physical qubits.  
Instead, it will compute with **logical qubits** built from many physical qubits.

### 3. Quantum memory

If we want to store a quantum state for a long time, we need to protect it from noise.

### 4. Reliable quantum gates

In fault-tolerant computing, gates are performed on encoded logical qubits.

### 5. Large-scale quantum computing

Without quantum error correction, noise destroys the computation before it becomes useful.

So the big practical idea is:

$
\text{physical qubits are noisy}
$

$
\text{logical qubits are protected}
$

$
\text{error correction makes long quantum computation possible}
$

# Very simple exercises

These exercises are intentionally easy.  
They are for understanding, not for difficulty.

## Exercise 1 — Majority vote

Change `corrupted_bits`.

Try:

$
000,\quad 010,\quad 111,\quad 101.
$

In [ ]:
corrupted_bits = "010"

print("corrupted bits:", corrupted_bits)
print("majority vote result:", majority_vote(corrupted_bits))

## Exercise 2 — Logical failure probability

Change `p`.

Question: when \(p\) is small, is the protected logical error smaller than the physical error?

In [ ]:
p = 0.05

physical_error = p
logical_error = repetition_failure_probability(3, p)

print("physical error:", physical_error)
print("logical error with 3-bit code:", logical_error)

## Exercise 3 — Encode a logical qubit

Change `a` and `b`.

The encoded logical state is:

$
a|000\rangle + b|111\rangle.
$

In [ ]:
a = 1
b = 1

encoded = encode_bit_flip_code(a, b)
print_state(encoded, basis_labels(3))

## Exercise 4 — Add a bit-flip error

Change `error_qubit` to `0`, `1`, or `2`.

In [ ]:
error_qubit = 0

encoded = encode_bit_flip_code(1, 1)
corrupted = apply_X_error(encoded, 3, error_qubit)

print("error qubit:", error_qubit)
print_state(corrupted, basis_labels(3))

## Exercise 5 — Read a syndrome

Change `bitstring`.

Try:

$
000,\quad 100,\quad 010,\quad 001.
$

In [ ]:
bitstring = "010"

syndrome = bit_flip_syndrome(bitstring)
correction = correction_from_syndrome(syndrome)

print("bitstring:", bitstring)
print("syndrome:", syndrome)
print("correction qubit:", correction)

## Exercise 6 — Correct an error

Change `error_qubit`.

In [ ]:
error_qubit = 2

encoded = encode_bit_flip_code(1, 1)
corrupted = apply_X_error(encoded, 3, error_qubit)

labels = basis_labels(3)
sample_bitstring = [
    label.strip("|⟩")
    for amp, label in zip(corrupted, labels)
    if abs(amp) > 1e-10
][0]

syndrome = bit_flip_syndrome(sample_bitstring)
correction = correction_from_syndrome(syndrome)

corrected = apply_X_error(corrupted, 3, correction)

print("error qubit:", error_qubit)
print("syndrome:", syndrome)
print("correction:", correction)
print("recovered:", np.allclose(corrected, encoded))

## Exercise 7 — Phase error after Hadamard

Run once with:

```python
apply_error = False
```

Then run again with:

```python
apply_error = True
```

Observe how the final measurement changes.

In [ ]:
apply_error = True

state = ket_plus

if apply_error:
    state = Z @ state

final = H @ state

print("applied phase error:", apply_error)
print_state(final, ["|0⟩", "|1⟩"])
show_probabilities(final, ["0", "1"], "Final probabilities")

## Final summary

Quantum error correction exists because:

$
\text{real qubits are noisy}
$

and

$
\text{quantum information is fragile.}
$

But we cannot simply copy or directly measure an unknown qubit.

So quantum error correction protects information by:

1. encoding one logical qubit into many physical qubits,
2. measuring syndromes,
3. detecting what error happened,
4. correcting the error,
5. preserving the logical quantum state.

The core idea is:

$
\text{measure the error, not the protected quantum information.}
$